# Rytmi — DSP + Gemma 4 rhythm-learning demo

**Problem:** dancers struggle to *hear* the beat in some music. Bachata's "1" is usually clear; kizomba's pulse is often subtle and the genre has no acoustic downbeat at all. Generic counting apps don't know the difference.

**Approach:** librosa-based DSP detects what the audio actually contains (tempo, beats, sections, beat-clarity per section, downbeat confidence). **Gemma 4** turns that grounded analysis into practical movement coaching — and refuses to claim things the analysis can't support. This notebook shows both halves of the pipeline on one kizomba and one bachata track.

## Pipeline

```
audio file ─► librosa DSP ─► RhythmAnalysis  ─► Gemma 4 prompt ─► coaching
             (beats,         (tempo, sections,    (style-aware,
              onsets,         downbeats,            grounded in
              HPSS,           beat-clarity,         analysis only)
              sections)       phases)
```

Hard rule: Gemma never invents musical facts. The prompt only knows what DSP reports, and per-style guards (e.g. "kizomba downbeat detection is out of scope") keep it honest about uncertainty.

In [ ]:
%matplotlib inline
import os
from pathlib import Path

from IPython.display import HTML, Markdown, display

from rytmi import analyze, load_audio
from rytmi import viz as rytmi_viz
from rytmi.dsp import describe_sections, describe_transitions, extract_transitions
from rytmi.llm import explain_rhythm, generate, load_cloud_model, polish_kizomba_tutor_output
from rytmi.prompts import (
    QUESTION_BACHATA_DRILLS,
    QUESTION_BACHATA_TUTOR,
    QUESTION_KIZOMBA_DRILLS,
    QUESTION_KIZOMBA_TRANSITIONS,
    QUESTION_KIZOMBA_TUTOR,
    QUESTION_LISTENING_GUIDE,
    QUESTION_RHYTHM_ANATOMY,
    QUESTION_SONG_ARC,
    downbeat_confidence_label,
    format_unified_timeline,
    verify_kizomba_drills_output,
    verify_kizomba_transitions_output,
)
from rytmi.vocal_activity import default_vocal_activity_source

# Local Gemma via Ollama is the project default. Override with
# RYTMI_API_BASE_URL / RYTMI_API_KEY env vars for any OpenAI-compatible endpoint.
CLOUD_BASE_URL = os.environ.get("RYTMI_API_BASE_URL", "http://localhost:11434/v1")
CLOUD_API_KEY = os.environ.get("RYTMI_API_KEY", "ollama")
CLOUD_MODEL_ID = os.environ.get("RYTMI_MODEL_ID", "google/gemma-4-26b-a4b-it")
#CLOUD_BASE_URL = "http://localhost:11434/v1"  # Ollama default
#CLOUD_MODEL_ID = "gemma4:e4b"                 # ~5 GB VRAM as Q4; use "gemma4:e2b" for smaller

EVAL = Path("../data/songs/eval_set")
KIZOMBA_TRACK = EVAL / "kizomba/Filomena_Maricoa_-_Teu_Toque_Official_Video [sKbWzD0mt6I].mp3"
BACHATA_TRACK = EVAL / "bachata/Romeo_Santos_-_Propuesta_Indecente_Official_Video [QFs3PIZb3js].mp3"

# Phase 38 vocal-activity envelope - mirrors notebook 05's plumbing. When True,
# a Demucs-based vocal source separates vocals per track and feeds the envelope
# into analyze(...) so the section labeler can carve `instrumental` sub-sections
# out of vocal-quiet, high-energy runs (`_relabel_vocal_drop_instrumentals` in
# dsp.py). Without this, vocal-drop runs collapse into generic `main` labels and
# the listening_guide / tutor lose the nuance. Flip to False on machines without
# Demucs / VRAM.
RUN_VOCAL_ACTIVITY = True

# Phase 40d - same-label transitions. When True, extract_transitions emits a
# Transition for every consecutive phase boundary (including same-label energy
# shifts like `main x4 medium -> main x2 high`). When False, only label-change
# boundaries appear (Phase 40 default behaviour). Flip to False if the
# same-label transitions read as cruft vs the section-role vocabulary already
# in kizomba_tutor's per-phase coaching.
INCLUDE_SAME_LABEL_TRANSITIONS = True

# Phase 40e — RUN_TRANSITION_RETRY=True asks Gemma to write any missing
# transition line one more time before the deterministic _fallback_transition_tail
# substitution kicks in. Honest with the project's signature pattern
# (code identifies, Gemma writes, code verifies) and reduces template
# repetition across same-label T# fills. Costs one extra Gemma call per
# missing boundary; flip to False to skip.
RUN_TRANSITION_RETRY = True

# Phase 41-lite — INCLUDE_PHASE_FEATURES=True appends qualitative texture
# (bass-driven / percussive / balanced from harm/perc ratios) and onset-density
# (sparse / steady / busy from onsets-per-beat) tags to each phase summary line
# in the analysis dump fed to Gemma. Read-only DSP, no schema change. Goal:
# more song-specific tutor and transition coaching. Flip to False to A/B.
INCLUDE_PHASE_FEATURES = False  # Phase 41-lite trial: tried, mixed result; off by default. Flip to True to revisit.

# Phase 41-D — INCLUDE_PHASE_VOCAL=True appends a per-phase vocal-presence
# tag (present / sparse / quiet) to the analysis dump fed to Gemma. The
# vocal_ratio per section is populated by analyze() when a vocal_env (Demucs)
# is provided. Vocals come and go more than HPSS texture does within a
# kizomba song, so this is the candidate signal most likely to give Gemma
# real inter-phase contrast. A/B trial — flip to False to compare.
INCLUDE_PHASE_VOCAL = False  # Phase 41-D trial: tried, same null result as 41-lite. See 41-2026-05-11 note.

processor, model = load_cloud_model(
    base_url=CLOUD_BASE_URL,
    api_key=CLOUD_API_KEY,
    model_id=CLOUD_MODEL_ID,
)

In [ ]:
print(CLOUD_BASE_URL)
print(CLOUD_MODEL_ID)

### Vocal-activity source (Phase 38)

Build a Demucs-based vocal source once so each track's `analyze(...)` call can
feed in a vocal-activity envelope. Without it, the section labeler can't carve
`instrumental` sub-sections out of vocal-quiet, high-energy runs (see
`_relabel_vocal_drop_instrumentals` in `src/rytmi/dsp.py`) — vocal-drop runs
collapse into plain `main`. Notebook 05 has had this since Phase 9; the demo
notebook gets it too as of Phase 38. The chained source falls back if Demucs is
unavailable; flip `RUN_VOCAL_ACTIVITY = False` to skip.

In [ ]:
vocal_source = None
if RUN_VOCAL_ACTIVITY:
    vocal_source = default_vocal_activity_source(prefer="demucs")
    print(f"[vocal-activity] configured source: {vocal_source!r}")
else:
    print("RUN_VOCAL_ACTIVITY=False; analyze() runs without vocal envelope "
          "(no `instrumental` sub-section labeling)")

## 1. Kizomba — the hard case

Kizomba has no acoustic downbeat. The pulse is slow (typically 50–95 BPM), often carried by bass and percussion that fade in and out across sections. A naive beat tracker locks onto whatever percussion is loudest and produces confidently-wrong markers in subtle sections.

Track: **Filomena Maricoa — Teu Toque** (clear-beat song with one severe break around 13s).

In [ ]:
kiz_audio = load_audio(KIZOMBA_TRACK)
kiz_vocal_env = None
if vocal_source is not None:
    try:
        kiz_vocal_env = vocal_source.compute(kiz_audio)
    except Exception as _e:
        print(f"[vocal-activity] kizomba FAILED ({_e}); continuing without envelope")
kiz = analyze(kiz_audio, dance_style="kizomba", vocal_env=kiz_vocal_env)

display(Markdown(
    f"**Tempo:** {kiz.tempo:.1f} BPM &nbsp; "
    f"**Beats:** {len(kiz.beats.times)} &nbsp; "
    f"**Sections:** {len(kiz.sections)} &nbsp; "
    f"**Duration:** {kiz.audio.duration:.1f}s &nbsp; "
    f"**vocal_env:** {'yes' if kiz_vocal_env is not None else 'no'}"
))

### Rhythm anatomy - what kizomba sounds like, broadly

Before zooming into a specific track, ask Gemma for a two-paragraph genre intro. `QUESTION_RHYTHM_ANATOMY` runs once per learning session and frames what the learner will hear in any kizomba track that follows: typical tempo range, time signature, what carries the pulse, structural arc, and major sub-style hints. It is **not track-specific** - the per-track listening guide and tutor below add the song's own detail. The kizomba downbeat guard applies, so the genre intro talks about syncopation and bass-led pulse without ever naming a "1".

In [ ]:
kiz_anatomy = explain_rhythm(kiz, QUESTION_RHYTHM_ANATOMY, processor, model)
print(kiz_anatomy)

### What librosa heard

Per-section table and an interactive timeline. The DSP layer reports **beat clarity per section** (subtle / moderate / clear) so Gemma can judge how confident to be. Notice that the `break` section has a low beat-clarity, even though the surrounding `main` sections are clear — the bass and percussion thin out together at the break.

In [ ]:
print(describe_sections(kiz))
display(HTML(rytmi_viz.interactive_timeline(
    kiz, title="Filomena Maricoa — Teu Toque (kizomba)", pixels_per_second=80,
)))

### Listening guide — orient the ear before the body

Before the song-arc narrative or any per-phase coaching, ask Gemma to walk the learner through the track as a **piece of music**. `QUESTION_LISTENING_GUIDE` produces two short paragraphs: an *orientation* (duration, tempo, structural arc, what carries the pulse) and a *difficulty map* (where the pulse will likely be hardest to follow, with one piece of listening advice for the hardest moment). It is explicitly forbidden from prescribing movement — that's the tutor's job. The point is to help the dancer **understand** the music first, then add the body.

In [ ]:
kiz_listening = explain_rhythm(kiz, QUESTION_LISTENING_GUIDE, processor, model)
print(kiz_listening)

### Song arc — the overall energy journey

Before zooming in to per-phase coaching, ask Gemma for a short narrative summary of the whole track. `QUESTION_SONG_ARC` is style-templated and never claims a downbeat — it describes the energy journey using the phase structure and timestamps from the analysis. This frames the per-phase tutor below: overview first, detail second.

In [ ]:
kiz_song_arc = explain_rhythm(kiz, QUESTION_SONG_ARC, processor, model)
print(kiz_song_arc)

### What Gemma teaches — kizomba tutor, one-pass and polished

`QUESTION_KIZOMBA_TUTOR` is a kizomba-specific prompt that turns the per-section beat-clarity labels into movement strategy. It is **explicitly forbidden** from naming a downbeat or `"1"` (kizomba doesn't have one); it must give recovery actions on `beat: subtle` sections; and on `beat: clear` breaks it prefers "reduce travel / keep a small pulse / reset / reconnect" over "pause and hold".

The output below shows two outputs side by side: the **one-pass draft** (what the prompt produces in a single LLM call — the documented baseline) and the **polished rewrite** (a second LLM call that preserves every P# header / time span / beat tag from the draft but tightens the coaching language against a stricter rubric). The polish is the **default for the demo recording** — non-deterministic generation occasionally slips in a "1 and 3" or a metric leak in the one-pass; polish catches those and produces consistently cleaner coaching. The one-pass is shown for transparency.

In [ ]:
kiz_draft = explain_rhythm(
    kiz, QUESTION_KIZOMBA_TUTOR, processor, model,
    include_phase_features=INCLUDE_PHASE_FEATURES,
    include_phase_vocal=INCLUDE_PHASE_VOCAL,
)
print(kiz_draft)

### Polished rewrite — the demo's recommended kizomba_tutor output

`polish_kizomba_tutor_output` runs a second LLM call against a stricter rubric, preserving every P# header / time span / beat tag from the draft above. **This is the recommended default for the demo recording.** It catches the rare slips the one-pass shows under non-determinism (Phase 33 listening-guide "1" leak, Phase 34 Isabelle "step on 1 and 3", Phase 34 Calo Pascoal "0.34" decimal narration) and produces consistently substantive coaching even on long similar-section tracks. Cost: a second LLM call per track.

In [ ]:
kiz_polished = polish_kizomba_tutor_output(
    kiz_draft, processor, model, track_name="Filomena — Teu Toque",
)
print(kiz_polished)

### Practice plan - drills tied to specific sections

Coaching tells the learner *what* to do; drills tell them *what to practice and for how long*. `QUESTION_KIZOMBA_DRILLS` asks Gemma for a per-section practice plan that **covers the whole song**: every phase maps to one drill line, and adjacent same-label same-beat phases may collapse into a single P# range (e.g. `P2-P5: main`). The raw Gemma draft is preserved for inspection, then `verify_kizomba_drills_output` normalizes the learner-facing output against DSP phases so ranges cannot cross section boundaries or duplicate a phase. This is the closing beat: overview, detail, then "now go practice."

In [ ]:
kiz_drills_raw = explain_rhythm(kiz, QUESTION_KIZOMBA_DRILLS, processor, model)
kiz_drills_verified = verify_kizomba_drills_output(kiz_drills_raw, kiz.phases)
kiz_drills = kiz_drills_verified.cleaned or kiz_drills_raw
kiz_drills_verified_stats = " ".join(
    f"{key}={kiz_drills_verified.stats.get(key, 0)}"
    for key in (
        "parsed",
        "repaired_ranges",
        "duplicate_phases",
        "filled_missing",
        "skipped_lines",
        "output_lines",
    )
)
print(kiz_drills)
print(f"\n[verifier: {kiz_drills_verified_stats}]")

### Transitions — what to do at the boundary (Phase 40)

The kizomba tutor and drills cover steady-state movement *during* each section. `QUESTION_KIZOMBA_TRANSITIONS` covers the moments *between* them — the anticipation cue (last 8-count of the incoming section), the boundary itself, and the re-entry cue (first 8-count of the outgoing section).

The split is the project's signature pattern: **code identifies the transitions, Gemma writes the coaching, code verifies the output**. `extract_transitions(phases)` walks the phase list and emits one `Transition` per label-change boundary (intro→main, main→break, break→main, main→outro, etc.) — same-label phase boundaries (energy-only changes) are skipped because the section-role vocabulary in `kizomba_tutor` already covers those. The transitions list lands in the prompt's analysis dump as a `Transitions (N label boundaries)` block, anchoring Gemma on real boundary times. `verify_kizomba_transitions_output` then drops any T# whose boundary doesn't match (within ±2s) and fills missing transitions with deterministic template text.

This closes the gap a learner hits when a `break` arrives mid-song without rehearsal — festival/YouTube dancers can plan transitions because they know the song; this output gives a regular learner a way to handle them on first listen.

In [ ]:
kiz_transitions = extract_transitions(
    kiz.phases, include_same_label=INCLUDE_SAME_LABEL_TRANSITIONS,
)
print(describe_transitions(kiz, include_same_label=INCLUDE_SAME_LABEL_TRANSITIONS))
print()
kiz_transitions_raw = explain_rhythm(
    kiz, QUESTION_KIZOMBA_TRANSITIONS, processor, model,
    include_same_label_transitions=INCLUDE_SAME_LABEL_TRANSITIONS,
    include_phase_features=INCLUDE_PHASE_FEATURES,
    include_phase_vocal=INCLUDE_PHASE_VOCAL,
)
kiz_transitions_verified = verify_kizomba_transitions_output(
    kiz_transitions_raw, kiz_transitions,
    retry_callback=(
        (lambda prompt: generate(processor, model, prompt, max_new_tokens=128))
        if RUN_TRANSITION_RETRY else None
    ),
)
kiz_transitions_text = kiz_transitions_verified.cleaned or kiz_transitions_raw
kiz_transitions_verified_stats = " ".join(
    f"{key}={kiz_transitions_verified.stats.get(key, 0)}"
    for key in (
        "parsed",
        "boundaries_matched",
        "boundaries_invented",
        "boundaries_missing_filled",
        "skipped_lines",
        "output_lines",
        "retried",
        "retry_succeeded",
    )
    if key in kiz_transitions_verified.stats
)
print(kiz_transitions_text)
print(f"\n[verifier: {kiz_transitions_verified_stats}]")

### Unified timeline — phases and transitions as one chronological story (Phase 40d)

`format_unified_timeline(tutor_text, transitions_text)` is a pure post-processing helper — no LLM call — that interleaves the polished kizomba_tutor's `P#` lines with the verified kizomba_transitions' `T#` lines in time order. The reader gets a single coherent narrative where each transition reads as the *bridge* into the next phase, instead of two parallel outputs sold side-by-side.

With `INCLUDE_SAME_LABEL_TRANSITIONS = True` (Phase 40d default), the timeline includes same-label energy/role shifts (e.g. `main ×4 medium → main ×2 high` between P3 and P4 on Filomena) — so the model has a chance to coach the building / sustaining / returning shifts the section-role vocabulary in `kizomba_tutor` already names. Flip the flag to `False` if those reads as cruft and you want only label-change boundaries (Phase 40 default).

In [ ]:
kiz_unified = format_unified_timeline(kiz_polished, kiz_transitions_text)
print(kiz_unified)

## 2. Bachata — same pipeline, inverted honesty

Bachata is the easier case. There **is** an acoustic "1" — typically marked by the güira pattern and the bongo-tumba — so the same DSP pipeline produces a meaningful downbeat confidence score, and Gemma can talk about phrase starts and the song's energy arc without the kizomba caveats.

Track: **Romeo Santos — Propuesta Indecente** (clear full break with RMS + onset-density drop).

The bachata flow runs the **same six-mode shape** as kizomba — `rhythm_anatomy → describe_sections + timeline → listening_guide → song_arc → bachata_tutor → bachata_drills (verified)` — but with **inverted honesty** about the "1": where `QUESTION_KIZOMBA_TUTOR` forbids naming a downbeat, `QUESTION_BACHATA_TUTOR` anchors the learner on the count when `downbeat_confidence` supports it and falls back to "trust the steady pulse, count internally" when it doesn't. The structural drills verifier (`verify_kizomba_drills_output`) is style-agnostic and serves both styles. No polish pass for bachata in this demo — the one-pass tutor output is what ships.

In [ ]:
bach_audio = load_audio(BACHATA_TRACK)
bach_vocal_env = None
if vocal_source is not None:
    try:
        bach_vocal_env = vocal_source.compute(bach_audio)
    except Exception as _e:
        print(f"[vocal-activity] bachata FAILED ({_e}); continuing without envelope")
bach = analyze(bach_audio, dance_style="bachata", vocal_env=bach_vocal_env)

conf = bach.downbeat_confidence
conf_str = (
    f"{downbeat_confidence_label(conf)} ({conf:.2f})" if conf is not None else "—"
)
display(Markdown(
    f"**Tempo:** {bach.tempo:.1f} BPM &nbsp; "
    f"**Beats:** {len(bach.beats.times)} &nbsp; "
    f"**Sections:** {len(bach.sections)} &nbsp; "
    f"**Downbeat \"1\":** {conf_str} &nbsp; "
    f"**vocal_env:** {'yes' if bach_vocal_env is not None else 'no'}"
))
print(describe_sections(bach))
display(HTML(rytmi_viz.interactive_timeline(
    bach, title="Romeo Santos — Propuesta Indecente (bachata)", pixels_per_second=80,
)))

### Rhythm anatomy — what bachata sounds like, broadly

Same `QUESTION_RHYTHM_ANATOMY` prompt, different `{style}` parameter. Bachata's genre intro should naturally reference the güira pattern and the bongo-tumba grounding the "1" — the kizomba downbeat guard does not apply for non-kizomba styles, so the model is free to talk about the acoustic "1" when it's actually present in the genre.

In [ ]:
bach_anatomy = explain_rhythm(bach, QUESTION_RHYTHM_ANATOMY, processor, model)
print(bach_anatomy)

### Listening guide — orient the ear before the body

Same `QUESTION_LISTENING_GUIDE` prompt as kizomba, different `{style}` parameter. The prompt is style-templated and forbids movement coaching for both styles. For bachata it should reference the güira / bongo-tumba grounding the pulse and use the `downbeat_confidence` band to frame how easy the "1" will be to find in the difficulty-map paragraph.

In [ ]:
bach_listening = explain_rhythm(bach, QUESTION_LISTENING_GUIDE, processor, model)
print(bach_listening)

### Song arc — the overall energy journey

Same `QUESTION_SONG_ARC` prompt as kizomba; bachata's energy progression maps to the same phase structure with the added bachata-specific texture (clear "1", güira / bongo-tumba) when the downbeat confidence supports it. The arc closes with one sentence on what makes this track distinct from typical bachata at the same tempo.

In [ ]:
bach_song_arc = explain_rhythm(bach, QUESTION_SONG_ARC, processor, model)
print(bach_song_arc)

### What Gemma teaches — bachata tutor (per-phase coaching)

`QUESTION_BACHATA_TUTOR` is the bachata equivalent of the kizomba tutor — same P# format, same beat-clarity-tag grounding, same recovery-vocabulary on `beat: subtle` sections — but with **inverted honesty** about the "1". Where the kizomba tutor explicitly forbids naming a downbeat (because detection is unreliable), the bachata tutor **anchors the learner on the count** when `downbeat_confidence` is in the high or moderate band, and falls back to "trust the steady pulse, count internally" when the confidence is low or absent. The vocabulary is bachata-specific: the 1-2-3-tap / 5-6-7-tap basic, the 8-count, "land the basic on 1" framing.

No polish pass for bachata in this demo — Phase 39 didn't add `polish_bachata_tutor_output`. If a future live run shows recurring slips on the one-pass output, polish becomes a follow-up.

In [ ]:
bach_tutor = explain_rhythm(bach, QUESTION_BACHATA_TUTOR, processor, model)
print(bach_tutor)

### Practice plan — drills tied to specific sections

Same shape as the kizomba drills above, with bachata vocabulary (1-2-3 / 5-6-7 anchoring) and bachata-specific phasing. `QUESTION_BACHATA_DRILLS` asks Gemma for a whole-song practice plan; the same `verify_kizomba_drills_output` symbol then validates each P# range against `bach.phases` — the verifier is style-agnostic by Phase 37c design and Phase 39 reused it for bachata without modification. The learner-facing output is normalized against DSP phases so ranges cannot cross section boundaries or duplicate a phase.

In [ ]:
bach_drills_raw = explain_rhythm(bach, QUESTION_BACHATA_DRILLS, processor, model)
bach_drills_verified = verify_kizomba_drills_output(bach_drills_raw, bach.phases)
bach_drills = bach_drills_verified.cleaned or bach_drills_raw
bach_drills_verified_stats = " ".join(
    f"{key}={bach_drills_verified.stats.get(key, 0)}"
    for key in (
        "parsed",
        "repaired_ranges",
        "duplicate_phases",
        "filled_missing",
        "skipped_lines",
        "output_lines",
    )
)
print(bach_drills)
print(f"\n[verifier: {bach_drills_verified_stats}]")

## What this demonstrates

- **Style-aware DSP.** `analyze(audio, dance_style=...)` runs a different beat-tracker path for kizomba (low-pass + low-band onset envelope tuned for batida) and reports per-style confidence (no downbeat for kizomba; explicit confidence band for bachata).
- **Grounded LLM coaching.** Every Gemma prompt is filled from the analysis only. Shared metric-guard and downbeat-guard helpers (Phase 35) live as module-level constants so when the wording needs to harden, every consuming prompt inherits the fix.
- **Listening before moving.** The `listening_guide` prompt orients the dancer to the music itself — what to hear, where it gets hard — before any per-phase coaching. Explicitly forbidden from prescribing movement; that frame is reserved for the tutor and drills below it.
- **Honest uncertainty.** When a section's beat-clarity is `subtle`, Gemma is told to suggest recovery actions (mark in place, count internally, re-enter at the next clear phase) rather than confidently prescribe steps.
- **Inverted-honesty parity across styles.** Kizomba and bachata share the same six-mode flow (`rhythm_anatomy → describe_sections → listening_guide → song_arc → tutor → drills`) but with **opposite stances on the "1"**: the kizomba tutor forbids naming a downbeat (detection unreliable), the bachata tutor anchors the learner on the count when `downbeat_confidence` supports it. The structural drills verifier (`verify_kizomba_drills_output`) is style-agnostic and serves both.
- **Polished kizomba tutor by default.** The polish pass runs by default for the demo because a second LLM call against a stricter rubric reliably catches the rare leaks one-pass generation shows under non-determinism. The one-pass output is kept visible for transparency. (No polish for bachata — the one-pass output is what ships.)

## Where to look next

- `notebooks/07_kizomba_tutor.ipynb` — runs the kizomba tutor on all 5 tap-reference tracks (incl. the `beat: subtle` honest-uncertainty case Charbel).
- `notebooks/06_kizomba_batida_check.ipynb` — visual diagnostic for the kizomba beat tracker.
- `notebooks/05_batch_analysis.ipynb` — every prompt in `ALL_QUESTIONS` over the eval set.
- `docs/project-vision.md` — what we are and aren't trying to build.
- `docs/how-it-works.md` — DSP architecture details.
- `docs/experiments/` — dated phase notes, including the kizomba mel-filterbank gotcha (Phase 28) and the tutor refresh (Phase 29 / 29b).

## Collect outputs for review

Final cell — gathers every Gemma output produced above into a single Markdown
dump and writes it to `00_demo_outputs.md` alongside the notebook. Useful
when iterating on prompts: re-run the notebook, then either copy the printed
block below or hand the output file to whoever is reviewing without making
them parse the full `.ipynb` (which carries large interactive timelines).

In [ ]:
OUTPUT_PATH = Path("00_demo_outputs.md")

_kiz_header = (
    f"tempo={kiz.tempo:.1f} BPM | beats={len(kiz.beats.times)} | "
    f"sections={len(kiz.sections)} | duration={kiz.audio.duration:.1f}s | "
    f"vocal_env={'yes' if kiz_vocal_env is not None else 'no'}"
)
_bach_header = (
    f"tempo={bach.tempo:.1f} BPM | beats={len(bach.beats.times)} | "
    f"sections={len(bach.sections)} | duration={bach.audio.duration:.1f}s | "
    f"downbeat_confidence={bach.downbeat_confidence:.2f} | "
    f"vocal_env={'yes' if bach_vocal_env is not None else 'no'}"
    if bach.downbeat_confidence is not None else
    f"tempo={bach.tempo:.1f} BPM | beats={len(bach.beats.times)} | "
    f"sections={len(bach.sections)} | duration={bach.audio.duration:.1f}s | "
    f"downbeat_confidence=None | "
    f"vocal_env={'yes' if bach_vocal_env is not None else 'no'}"
)

_sections = [
    ("kizomba - rhythm_anatomy (genre intro)", kiz_anatomy),
    ("Filomena Maricoa - Teu Toque (kizomba) - describe_sections",
     f"{_kiz_header}\n\n{describe_sections(kiz)}"),
    ("Filomena Maricoa - Teu Toque (kizomba) - listening_guide", kiz_listening),
    ("Filomena Maricoa - Teu Toque (kizomba) - song_arc", kiz_song_arc),
    ("Filomena Maricoa - Teu Toque (kizomba) - kizomba_tutor (one-pass)", kiz_draft),
    ("Filomena Maricoa - Teu Toque (kizomba) - kizomba_tutor (polished)", kiz_polished),
    ("Filomena Maricoa - Teu Toque (kizomba) - kizomba_drills (verified)", kiz_drills),
    (
        "Filomena Maricoa - Teu Toque (kizomba) - kizomba_drills verifier stats",
        kiz_drills_verified_stats,
    ),
    (
        "Filomena Maricoa - Teu Toque (kizomba) - kizomba_drills raw Gemma draft",
        kiz_drills_raw,
    ),
    (
        "Filomena Maricoa - Teu Toque (kizomba) - describe_transitions",
        describe_transitions(kiz, include_same_label=INCLUDE_SAME_LABEL_TRANSITIONS),
    ),
    (
        "Filomena Maricoa - Teu Toque (kizomba) - kizomba_transitions (verified)",
        kiz_transitions_text,
    ),
    (
        "Filomena Maricoa - Teu Toque (kizomba) - kizomba_transitions verifier stats",
        kiz_transitions_verified_stats,
    ),
    (
        "Filomena Maricoa - Teu Toque (kizomba) - kizomba_transitions raw Gemma draft",
        kiz_transitions_raw,
    ),
    (
        "Filomena Maricoa - Teu Toque (kizomba) - unified timeline (phases + transitions)",
        kiz_unified,
    ),
    ("bachata - rhythm_anatomy (genre intro)", bach_anatomy),
    ("Romeo Santos - Propuesta Indecente (bachata) - describe_sections",
     f"{_bach_header}\n\n{describe_sections(bach)}"),
    ("Romeo Santos - Propuesta Indecente (bachata) - listening_guide", bach_listening),
    ("Romeo Santos - Propuesta Indecente (bachata) - song_arc", bach_song_arc),
    ("Romeo Santos - Propuesta Indecente (bachata) - bachata_tutor", bach_tutor),
    ("Romeo Santos - Propuesta Indecente (bachata) - bachata_drills (verified)", bach_drills),
    (
        "Romeo Santos - Propuesta Indecente (bachata) - bachata_drills verifier stats",
        bach_drills_verified_stats,
    ),
    (
        "Romeo Santos - Propuesta Indecente (bachata) - bachata_drills raw Gemma draft",
        bach_drills_raw,
    ),
]

_body = f"# 00_demo outputs - model={CLOUD_MODEL_ID}\n\n"
_body += "\n\n".join(f"## {name}\n\n```\n{out.strip()}\n```" for name, out in _sections)

OUTPUT_PATH.write_text(_body)
print(_body)
print(f"\n[wrote {len(_body):,} chars to {OUTPUT_PATH.resolve()}]")